# End of week 1 exercise


To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!


In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# 1. Load your credentials
load_dotenv()

# 2. Configure the OpenRouter Client
# OpenRouter requires /api/v1 at the end of the base_url
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

# 3. Define the question and prompts
question = """
Explain the difference between a list comprehension and a generator expression in Python. 
When should I use one over the other?
"""

system_prompt = "You are a helpful technical tutor who answers questions about software engineering, Python, and data science."
user_prompt = f"Please provide a detailed explanation for this question: {question}"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# 4. Get the answer with streaming
print("Tutor is generating your explanation...\n")

try:
    # Using a fast, reliable model from OpenRouter
    stream = client.chat.completions.create(
        model="google/gemini-2.0-flash-001", 
        messages=messages,
        stream=True,
        extra_headers={
            "HTTP-Referer": "http://localhost:3000", # Optional for OpenRouter
            "X-Title": "Technical Tutor Tool",       # Optional for OpenRouter
        }
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
    # Safely check if the chunk has choices and if those choices have a delta
      if chunk.choices and chunk.choices[0].delta.content is not None:
        content = chunk.choices[0].delta.content
        response += content
        # Update the display
        update_display(Markdown(response), display_id=display_handle.display_id)


except Exception as e:
    print(f"❌ Error: {e}")
